# Facility temperature — the weather signal, and what it does to demand

`facility-temperature-*` is AEMO's per-facility **maximum daily temperature**: 55,727 rows over 61
facility codes and 973 days, 2 January 2024 to 31 August 2026. It is the only exogenous driver in
this project — every other series is the market observing itself — and it is the smallest and
dirtiest file in the set.

It is also the file where the data-quality story is *not* about holes. `docs/DATA_NOTES.md` opens by
saying this is unusually clean market data. That claim does not survive contact with this file.

Three findings run through the notebook:

- **Nine of the 61 facility codes do not report temperature at all.** Six sit at exactly 41.0 every
  day on record; three report a negative sentinel around -1.4. They are numeric and non-null, so
  `notna()` passes them straight through into any average that touches them.
- **The 50 real series are only 21 weather stations.** Ten facility codes share one Pinjar/Neerabup
  signal. Treating facility columns as independent observations multiply-counts a handful of
  instruments.
- **Temperature explains daily peak demand, but only once you stop fitting a line to it.** Pearson's
  r is 0.27 — apparently useless. Break the relationship at an empirically located balance point of
  27.0 degC and R-squared goes to 0.72. The linear correlation is not weak evidence of a weak
  relationship; it is a strong relationship measured with the wrong instrument.

The notebook ends where it can no longer go: daily maxima cannot reach the 5-minute demand series,
so peak *timing* is out of scope without an external hourly source.

In [1]:
# Load temperature, and the series it will be tested against
import sys
sys.path.insert(0, '../src')  # Add src directory to path (go up one level from notebooks/)
from wa_data import (load_temperature, temperature_diagnostics, temperature_stations,
                     temperature_wide, load_demand, load_price, load_facility_scada,
                     WA_RECORD_MAX_C, PLACEHOLDER_CORR)

# Load with explicit path (go up one level from notebooks/)
t_all = load_temperature(raw='../data/raw', drop_placeholders=False)   # everything, as published
t = load_temperature(raw='../data/raw')                                # placeholders removed

In [2]:
print(t_all.head())
print(t_all.shape)
print(t_all.columns.tolist())
print(f"\nfacility codes: {t_all.facility_code.nunique()}   "
      f"days: {t_all.date.nunique()}   "
      f"coverage: {t_all.date.min():%Y-%m-%d} -> {t_all.date.max():%Y-%m-%d}")

        date  facility_code  temp_max_c  implausible
0 2024-01-02      ALCOA_WGP        28.9        False
1 2024-01-02  ALINTA_PNJ_U1        30.2        False
2 2024-01-02  ALINTA_PNJ_U2        30.2        False
3 2024-01-02  ALINTA_WGP_GT        27.5        False
4 2024-01-02  ALINTA_WGP_U2        27.5        False
(53243, 4)
['date', 'facility_code', 'temp_max_c', 'implausible']

facility codes: 59   days: 973   coverage: 2024-01-02 -> 2026-08-31


In [3]:
# QUIRK 8, checked rather than trusted. The claim is that some series are registered
# constants rather than measurements. The test does not look for specific values: a
# genuine SWIS weather series must track the fleet-wide daily median, because the whole
# footprint shares one seasonal cycle. Anything that does not, is not weather.
import numpy as np
import pandas as pd

diag = temperature_diagnostics(raw='../data/raw')
print("verdict counts:")
print(diag.verdict.value_counts().to_string(), f"   (of {len(diag)} facility codes)\n")

print("NON-WEATHER SERIES - the evidence:")
print(diag[diag.verdict != 'weather'][
    ['n', 'n_distinct', 'std', 'corr_fleet', 'min', 'median', 'max', 'verdict']
].round(3).to_string())

# The separation is not a tuning choice. There is no series anywhere near the threshold.
gen = diag[diag.verdict == 'weather'].corr_fleet
print(f"\nweather series correlate {gen.min():.3f} to {gen.max():.3f} with the fleet median.")
print(f"the sentinel series scores {diag.loc['MUNGARRA_GT1', 'corr_fleet']:.3f}; "
      f"the constants have zero variance, so it is undefined.")
print(f"threshold PLACEHOLDER_CORR = {PLACEHOLDER_CORR} sits in an empty gap "
      f"{gen.min() - diag.loc['MUNGARRA_GT1', 'corr_fleet']:.2f} wide.")

# Why notna() is not enough: the sentinel is numeric, non-null, and 968 of 973 days.
sent = t_all[t_all.facility_code == 'MUNGARRA_GT1']
print(f"\nMUNGARRA_GT1: {sent.temp_max_c.notna().sum()} non-null readings, of which "
      f"{sent.temp_max_c.isin([-1.401, -1.64, -1.636]).sum()} are one of three sentinel values.")
print("monthly medians (a real Geraldton-region series would peak in Jan/Feb):")
print(sent.groupby(sent.date.dt.month).temp_max_c.median().round(2).to_string())

verdict counts:
verdict
weather     50
constant     6
sentinel     3
empty        2    (of 61 facility codes)

NON-WEATHER SERIES - the evidence:
                          n  n_distinct    std  corr_fleet    min  median     max   verdict
ALINTA_WWF              973           1  0.000         NaN  41.00  41.000  41.000  constant
BADGINGARRA_WF1         973           1  0.000         NaN  41.00  41.000  41.000  constant
MERSOLAR_PV1            973           1  0.000         NaN  41.00  41.000  41.000  constant
SBSOLAR1_CUNDERDIN_PV1  788           1  0.000         NaN  41.00  41.000  41.000  constant
WARRADARGE_WF1          973           1  0.000         NaN  41.00  41.000  41.000  constant
YANDIN_WF1              973           1  0.000         NaN  41.00  41.000  41.000  constant
ALBANY_WF1                0           0    NaN         NaN    NaN     NaN     NaN     empty
GRASMERE_WF1              0           0    NaN         NaN    NaN     NaN     NaN     empty
GREENOUGH_RIVER_PV1     97

In [4]:
# QUIRK 9: temperature is reported per FACILITY but measured per STATION.
st = temperature_stations(raw='../data/raw')
w = temperature_wide(raw='../data/raw')

print(f"{len(st)} genuine facility series collapse to {st.nunique()} distinct stations.\n")
grp = st.groupby(st).size().sort_values(ascending=False)
for s, n in grp.items():
    members = sorted(st.index[st == s])
    shown = ', '.join(members[:4]) + (f' ... (+{len(members) - 4})' if len(members) > 4 else '')
    print(f"  {s:<26} {n:2d} facilities   {shown}")

# Two facilities in the same group are identical, not merely similar.
pin = sorted(st.index[st == st['PINJAR_GT1']])
a, b = w[pin[0]], w[pin[1]]
print(f"\n{pin[0]} vs {pin[1]}: max abs difference "
      f"{(a - b).abs().max():.10f} over {int((a.notna() & b.notna()).sum())} shared days.")

# Completeness, on the STATION grid. Row counts are the wrong unit here because the
# facility count changes as units commission.
days = pd.date_range(t.date.min(), t.date.max(), freq='D')
print(f"\ncalendar days spanned: {len(days)}   days with any reading: {t.date.nunique()}")
miss = sorted(set(days) - set(t.date.unique()))
print(f"missing days: {len(miss)}" +
      (f"   first few: {[str(d.date()) for d in miss[:5]]}" if miss else ""))

# Readings above the WA record are FLAGGED, never clipped.
imp = t[t.implausible]
print(f"\nreadings above the WA record of {WA_RECORD_MAX_C} C: {len(imp)} "
      f"across {imp.facility_code.nunique()} facilities")
print(imp.groupby('facility_code').temp_max_c.agg(['size', 'max'])
      .sort_values('max', ascending=False).head(3).to_string())

50 genuine facility series collapse to 21 distinct stations.

  S01_NEWGEN_NEERABUP_GT1    10 facilities   NEWGEN_NEERABUP_GT1, PINJAR_GT1, PINJAR_GT10, PINJAR_GT11 ... (+6)
  S02_ALINTA_WGP_GT           7 facilities   ALINTA_WGP_GT, ALINTA_WGP_U2, BW1_BLUEWATERS_G2, BW2_BLUEWATERS_G1 ... (+3)
  S03_FLATROCKS_WF1           6 facilities   FLATROCKS_WF1, PRK_AG, STHRNCRS_EG, TESLA_NORTHAM_G1 ... (+2)
  S04_COCKBURN_CCG1           4 facilities   COCKBURN_CCG1, KWINANA_GT2, KWINANA_GT3, TIWEST_COG1
  S05_ALINTA_PNJ_U1           2 facilities   ALINTA_PNJ_U1, ALINTA_PNJ_U2
  S06_INVESTEC_COLLGAR_WF1    2 facilities   INVESTEC_COLLGAR_WF1, NAMKKN_MERR_SG1
  S07_KEMERTON_GT11           2 facilities   KEMERTON_GT11, KEMERTON_GT12
  S08_MWF_MUMBIDA_WF1         2 facilities   MWF_MUMBIDA_WF1, TESLA_GERALDTON_G1
  S09_NEWGEN_KWINANA_CCG1     2 facilities   NEWGEN_KWINANA_CCG1, PERTHENERGY_KWINANA_GT1
  S10_TESLA_KEMERTON_G1       2 facilities   TESLA_KEMERTON_G1, TESLA_PICTON_G1
  S11_ALCOA_WGP   

## Visualisation

Six views, ordered instrument -> weather -> demand -> money:

1. **What is actually a measurement** — the four verdict groups, drawn as the series they are.
2. **Twenty-one stations, one seasonal cycle** — the annual cycle and how far apart the state's
   corners sit on any given day.
3. **The U-curve, and the balance point** — the centrepiece: why Pearson's r is the wrong tool.
4. **Heatwaves compound** — the same temperature costs more on day three than on day one.
5. **Where the price tail lives** — spike probability by temperature, with the cap change respected.
6. **Storage on hot days** — the hypothesis that did not survive.

Colour carries **temperature** throughout, on a diverging scale centred at the balance point found
in chart 3: blue below it, orange above it, neutral grey at it. That is not decoration — below the
balance point the building stock is heating and above it it is cooling, so the midpoint is a real
physical boundary rather than a convenient middle of the data range.

In [5]:
# ── Chart setup: palette, shared theme, and derived frames ───────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Chart chrome, light surface — identical to the demand, DPV, price and SCADA
# notebooks, so all five read as one system.
SURFACE, INK, INK_2, MUTED, GRID, AXIS = (
    "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7")

BLUE, ORANGE, RED, AQUA = "#2a78d6", "#eb6834", "#e34948", "#1baf7a"

# TEMPERATURE is the semantic axis of this notebook, so it takes the DIVERGING
# pair, keeping the meaning the two hues carry in the SCADA notebook: blue cool,
# orange warm, NEUTRAL GRAY at the midpoint — never a hue — so the balance point
# reads as the boundary it is. Validated as a pair on this surface: worst CVD
# deltaE 24.7 (protan), normal-vision deltaE 33.6, both clear.
GRAY_MID = "#f0efec"
ORANGE_RAMP = ["#fbe0d4", "#f7c0a3", "#f39d73", "#eb6834", "#c94e1f", "#a03c17"]
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95"]
DIVERGING = [[i / 12, c] for i, c in
             enumerate(BLUE_RAMP[::-1] + [GRAY_MID] + ORANGE_RAMP)]

# VERDICT is categorical with four values, one of which ("weather") is the subject
# and three of which are defects. The defects take the warning hues and are
# DIRECT-LABELLED; weather takes blue. Colour never carries the meaning alone.
VERDICT_COLOR = {"weather": BLUE, "constant": ORANGE, "sentinel": RED, "empty": MUTED}

# YEAR is ordinal, so it keeps the one-hue ramp from the price and SCADA notebooks:
# the later the year, the darker the line.
YEAR_COLOR = {2024: "#3987e5", 2025: "#256abf", 2026: "#0d366b"}
PARTIAL = {2024: "from 2 Jan", 2026: "to 31 Aug"}
YEARS = [2024, 2025, 2026]
LABEL = {y: (f"{y} ({PARTIAL[y]})" if y in PARTIAL else str(y)) for y in YEARS}
DASH = {y: ("dot" if y in PARTIAL else "solid") for y in YEARS}

MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


def style(fig, title, subtitle=None, height=420, hover="x unified",
          top=108, bottom=58, legend_y=1.0, showlegend=True):
    """Shared theme: light surface, recessive grid, muted axes, ink-coloured text."""
    head = f"<b>{title}</b>"
    if subtitle:
        head += f"<br><span style='font-size:12.5px;color:{INK_2}'>{subtitle}</span>"
    fig.update_layout(
        title=dict(text=head, font=dict(size=17, color=INK), x=0, xanchor="left", y=0.97),
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, hovermode=hover,
        font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", size=12, color=INK_2),
        height=height, margin=dict(t=top, r=30, b=bottom, l=74), showlegend=showlegend,
        legend=dict(orientation="h", yanchor="bottom", y=legend_y, xanchor="left", x=0,
                    bgcolor="rgba(0,0,0,0)", font=dict(size=11.5)))
    fig.update_xaxes(gridcolor=GRID, zeroline=False, linecolor=AXIS,
                     ticks="outside", tickcolor=AXIS, tickfont=dict(color=MUTED))
    fig.update_yaxes(gridcolor=GRID, zeroline=False, linecolor=AXIS,
                     ticks="outside", tickcolor=AXIS, tickfont=dict(color=MUTED))
    for a in (fig.layout.annotations or []):
        if a.font is None or a.font.size is None:
            a.font = dict(size=12.5, color=INK_2)
    return fig


# ── derived frames used by more than one chart ───────────────────────────────
# The reference station. PINJAR/NEERABUP is the largest group (10 facility codes)
# and sits in the Perth metropolitan load centre, which is where the demand is.
# STATED ASSUMPTION: it stands in for "Perth weather" throughout, and chart 3's
# comparability checks re-run the headline fit on every other station.
REF_STATION = st['PINJAR_GT1']
REF_FACS = sorted(st.index[st == REF_STATION])
perth = w['PINJAR_GT1'].rename('tmax')

# Station-level wide frame: one column per station, not per facility.
wst = w.rename(columns=st)
wst = wst.T.groupby(level=0).first().T

# Daily demand, joined to the reference station.
dem = load_demand(raw='../data/raw').set_index('ts')
day = pd.DataFrame({
    'tmax': perth,
    'peak': dem.operational_demand_mw.resample('D').max(),
    'unsch_peak': dem.unscheduled_demand_mw.resample('D').max(),
    'min_unsch': dem.unscheduled_demand_mw.resample('D').min(),
}).dropna()
day['year'] = day.index.year
day['month'] = day.index.month
day['dow'] = day.index.dayofweek
day['is_weekend'] = day.dow >= 5

print(f"reference station {REF_STATION}: {len(REF_FACS)} facility codes")
print(f"station-level frame: {wst.shape[0]} days x {wst.shape[1]} stations")
print(f"demand join: {len(day)} days "
      f"({day.index.min():%Y-%m-%d} -> {day.index.max():%Y-%m-%d})")
print(f"  weekdays {int((~day.is_weekend).sum())}, weekends {int(day.is_weekend.sum())}")

reference station S01_NEWGEN_NEERABUP_GT1: 10 facility codes
station-level frame: 973 days x 21 stations
demand join: 966 days (2024-01-02 -> 2026-08-31)
  weekdays 690, weekends 276


### 1. What is actually a measurement

Four groups, drawn as the series they are. The point of the chart is that the defects are not subtle
once plotted and completely invisible in a summary statistic: a mean over all 61 facility codes is a
mean over six flat 41.0 lines and three flat -1.4 lines.

The **41.0** group is the more dangerous of the two, because 41 degC is a *plausible* WA summer
maximum. It fails only on variance — the same value on all 973 days, including every day in July.
The **-1.4** group fails on sign and on season. Both are almost certainly registered reference
values for the facility rather than readings from an instrument.

`ALBANY_WF1` and `GRASMERE_WF1` are honestly empty: 973 rows, no values. They cost nothing, because
`notna()` does catch them.

In [6]:
# ── Chart 1: the four verdict groups, drawn as series ────────────────────────
reps = {'weather': 'PINJAR_GT1', 'constant': 'YANDIN_WF1',
        'sentinel': 'MUNGARRA_GT1', 'empty': 'ALBANY_WF1'}
wall = temperature_wide(raw='../data/raw', drop_placeholders=False)

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.14, row_heights=[0.62, 0.38],
                    subplot_titles=["One representative series per verdict, as published",
                                    "Facility codes by verdict"])

for v, f in reps.items():
    if f not in wall.columns:
        continue
    s = wall[f].dropna()
    if s.empty:
        continue
    fig.add_trace(go.Scatter(
        x=s.index, y=s.values, name=f"{v} — {f}", mode='lines',
        line=dict(color=VERDICT_COLOR[v], width=1.3 if v == 'weather' else 2.2),
        opacity=0.9), row=1, col=1)
    # Direct label, so the reading never depends on matching a colour to a key.
    fig.add_annotation(x=s.index[-1], y=s.iloc[-1], text=f"  {v}", showarrow=False,
                       xanchor='left', font=dict(size=11.5, color=VERDICT_COLOR[v]),
                       row=1, col=1)

cnt = diag.verdict.value_counts().reindex(['weather', 'constant', 'sentinel', 'empty']).fillna(0)
fig.add_trace(go.Bar(x=list(cnt.index), y=list(cnt.values),
                     marker_color=[VERDICT_COLOR[v] for v in cnt.index],
                     text=[f"{int(v)}" for v in cnt.values], textposition='outside',
                     showlegend=False, cliponaxis=False), row=2, col=1)

fig.update_yaxes(title_text="max daily temp (°C)", row=1, col=1)
fig.update_yaxes(title_text="facility codes", row=2, col=1, range=[0, 58])
fig.update_xaxes(range=[wall.index.min(), wall.index.max() + pd.Timedelta(days=110)], row=1, col=1)
style(fig, "Nine of 61 facility codes do not report temperature",
      "Six constant at 41.0 °C, three on a −1.4 °C sentinel, two empty. Every one of the nine is "
      "numeric and passes a null check.",
      height=620, hover='closest', legend_y=1.02)
fig.show()

# The two defects push in OPPOSITE directions, so the headline bias is small and
# the aggregate looks harmless. That is the trap, not a reassurance: a fleet mean
# is one of the few statistics in which they partly cancel.
hi = t_all[t_all.facility_code.isin(diag.index[diag.verdict == 'constant'])].temp_max_c
lo = t_all[t_all.facility_code.isin(diag.index[diag.verdict == 'sentinel'])].temp_max_c
print("A mean over all facility codes, versus a mean over the real ones:")
print(f"  all 61 codes : {t_all.temp_max_c.mean():.2f} °C  ({len(t_all):,} readings)")
print(f"  50 weather   : {t.temp_max_c.mean():.2f} °C  ({len(t):,} readings)")
print(f"  net bias     : {t_all.temp_max_c.mean() - t.temp_max_c.mean():+.2f} °C")
print("")
print("the net is small only because the defects oppose each other:")
print(f"  {len(hi):,} readings pinned at 41.0 °C pull the mean UP")
print(f"  {len(lo):,} readings on the −1.4 °C sentinel pull it DOWN")
print("any per-facility or per-station statistic gets the full error, uncancelled:")
for f in ['YANDIN_WF1', 'MUNGARRA_GT1']:
    v = t_all.loc[t_all.facility_code == f, 'temp_max_c'].mean()
    print(f"  {f:<16} reports {v:6.2f} °C mean against a fleet median of "
          f"{t.temp_max_c.median():.1f} °C")

A mean over all facility codes, versus a mean over the real ones:
  all 61 codes : 25.60 °C  (53,243 readings)
  50 weather   : 25.34 °C  (44,458 readings)
  net bias     : +0.26 °C

the net is small only because the defects oppose each other:
  5,866 readings pinned at 41.0 °C pull the mean UP
  2,919 readings on the −1.4 °C sentinel pull it DOWN
any per-facility or per-station statistic gets the full error, uncancelled:
  YANDIN_WF1       reports  41.00 °C mean against a fleet median of 24.5 °C
  MUNGARRA_GT1     reports  -1.36 °C mean against a fleet median of 24.5 °C


### 2. Twenty-one stations, one seasonal cycle

The left panel is the annual cycle at each of the 21 stations: monthly median maximum, southern
hemisphere, so the peak is January–February and the trough is July. Every station traces the same
shape, which is the fact that made the verdict test in chart 1 work.

The right panel is the part that matters for forecasting: the **spread across the state on the same
day**. The SWIS spans roughly 1,000 km, and on a given day its corners are far apart. A single
"system temperature" is therefore a real simplification, and the chart says how big a one.

Two exclusions, both stated rather than silent. The 21 readings above the WA record are dropped from
the spread — a spread is a maximum minus a minimum, so the one 62.8 °C sensor fault would set the
headline figure by itself and report a 41.5 °C spread that never happened. And the warmest/coolest
ranking uses only the 12 stations covering at least 900 days, because three battery sites commission
partway through the record and would rank artificially cool for having missed two summers.

Station names are `S<n>_<representative facility>`. AEMO publishes no station identifier, so naming
them after a member facility is traceable, where inventing place names would not be.

In [7]:
# ── Chart 2: seasonal cycle by station, and same-day spread ──────────────────
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.09, column_widths=[0.56, 0.44],
                    subplot_titles=["Annual cycle, monthly median by station",
                                    "Spread across stations on the same day"])

# The 21 flagged readings above the WA record are sensor faults, and a spread is a
# max-minus-min, so a single bad value would set the headline number by itself.
# They are excluded HERE and only here; the loader still returns them flagged.
wsc = wst.mask(wst > WA_RECORD_MAX_C)

# Ranking stations by mean temperature is only fair between stations that cover
# the same days. Three ESR sites start partway through the record and would rank
# "cool" purely for having missed two summers.
COV = 900
cov = wsc.notna().sum()
FULL = list(cov[cov >= COV].index)

mm = wsc.groupby(wsc.index.month).median()
warm = wsc[FULL].mean().sort_values()         # colour by how warm the station is
norm = ((wsc.mean() - warm.min()) / (warm.max() - warm.min())).clip(0, 1)


def temp_hue(u):
    """Sample the diverging ramp; u in [0, 1]."""
    idx = int(round(u * (len(DIVERGING) - 1)))
    return DIVERGING[idx][1]


for s in wsc.columns:
    fig.add_trace(go.Scatter(x=MONTHS, y=mm[s].values, mode='lines', name=s,
                             line=dict(color=temp_hue(norm[s]), width=1.6),
                             opacity=0.85, showlegend=False,
                             hovertemplate=f"{s}<br>%{{x}}: %{{y:.1f}} °C<extra></extra>"),
                  row=1, col=1)

# Direct-label the extremes only; 21 keys would be unreadable.
for s, tag in [(warm.index[-1], "warmest"), (warm.index[0], "coolest")]:
    fig.add_annotation(x=11, y=mm[s].iloc[11], text=f"  {s.split('_', 1)[1]} ({tag})",
                       showarrow=False, xanchor='left', xref='x', yref='y',
                       font=dict(size=10.5, color=temp_hue(norm[s])), row=1, col=1)

spread = (wsc.max(axis=1) - wsc.min(axis=1)).dropna()
fig.add_trace(go.Histogram(x=spread.values, nbinsx=44, marker_color=BLUE, opacity=0.85,
                           showlegend=False,
                           hovertemplate="spread %{x:.0f} °C<br>%{y} days<extra></extra>"),
              row=1, col=2)
fig.add_vline(x=float(spread.median()), line=dict(color=INK, width=1.4, dash='dot'),
              annotation_text=f" median {spread.median():.1f} °C",
              annotation_position="top right",
              annotation_font=dict(size=11, color=INK), row=1, col=2)

fig.update_yaxes(title_text="median max daily temp (°C)", row=1, col=1)
fig.update_xaxes(range=[-0.4, 14.6], row=1, col=1)
fig.update_yaxes(title_text="days", row=1, col=2)
fig.update_xaxes(title_text="warmest station − coolest station, same day (°C)", row=1, col=2)
style(fig, "One seasonal cycle, but not one temperature",
      f"21 stations, {len(spread)} days, {len(FULL)} of them covering the full record. Every "
      f"station shares the shape; on the median day they still span {spread.median():.1f} °C, "
      f"and on the widest {spread.max():.0f} °C.",
      height=460, hover='closest', showlegend=False)
fig.show()

print(f"same-day spread across stations: median {spread.median():.1f} °C, "
      f"p95 {spread.quantile(.95):.1f} °C, max {spread.max():.1f} °C")
print(f"  (flagged readings above {WA_RECORD_MAX_C} °C excluded; including the single "
      f"62.8 °C fault would report a spurious 41.5 °C maximum spread)")
print("")
print(f"ranked over the {len(FULL)} stations with >= {COV} days:")
print(f"  warmest: {warm.index[-1]} ({warm.iloc[-1]:.1f} °C)")
print(f"  coolest: {warm.index[0]} ({warm.iloc[0]:.1f} °C)")
print(f"  the {int(wsc.shape[1] - len(FULL))} short-coverage stations are excluded from the "
      f"ranking; they commission mid-record and would rank cool for missing summers.")

same-day spread across stations: median 7.8 °C, p95 14.4 °C, max 20.7 °C
  (flagged readings above 50.7 °C excluded; including the single 62.8 °C fault would report a spurious 41.5 °C maximum spread)

ranked over the 12 stations with >= 900 days:
  warmest: S08_MWF_MUMBIDA_WF1 (27.3 °C)
  coolest: S10_TESLA_KEMERTON_G1 (23.4 °C)
  the 9 short-coverage stations are excluded from the ranking; they commission mid-record and would rank cool for missing summers.


### 3. The U-curve, and the balance point

This is the chart the notebook exists for.

Daily maximum temperature against daily peak operational demand, weekdays only. The linear
correlation is **r = 0.27**, which on its own would retire the variable. The scatter says why that
number is meaningless: the relationship is a **U**. Demand rises when it is hot because of air
conditioning and rises when it is cold because of resistive heating, and a straight line through a U
measures the difference between two opposite slopes rather than the strength of either.

The fit is the standard degree-day decomposition, with one difference: the **balance point is
located from the data** rather than assumed. Northern-hemisphere convention puts it at 18 degC, which
is simply wrong here — the search over 14 to 30 degC in quarter-degree steps puts the SSE minimum at
**27.0 degC**. Below it, each degree colder adds about 81 MW to the peak; above it, each degree
hotter adds about 92 MW.

Two honest caveats sit on this fit. The cold arm rests on thin data — Perth barely gets cold, and
only four weekdays in three years fall below 15 degC — so the heating slope is the less certain of the
two, and the fit should not be extrapolated below about 12 degC. And temperature is not the only thing
that moved over three years: the comparability checks at the end separate the weather response from
the trend, and refit the balance point on every other station, where it proves less stable than a
single fit suggests.

In [8]:
# ── Chart 3: the U-curve and the fitted balance point ────────────────────────
# Weekdays only. Weekend demand is a different regime (chart 3b below tests that),
# and mixing the two would let the day-of-week effect leak into the temperature one.
wd = day[~day.is_weekend].copy()


def dd_fit(frame, bp):
    """Least-squares peak ~ 1 + CDD + HDD at a given balance point."""
    cdd = np.maximum(frame.tmax - bp, 0.0)
    hdd = np.maximum(bp - frame.tmax, 0.0)
    X = np.column_stack([np.ones(len(frame)), cdd, hdd])
    coef, *_ = np.linalg.lstsq(X, frame.peak.values, rcond=None)
    resid = X @ coef - frame.peak.values
    sse = float((resid ** 2).sum())
    sst = float(((frame.peak - frame.peak.mean()) ** 2).sum())
    return coef, sse, 1 - sse / sst


# Locate the balance point by exhaustive search rather than convention.
grid = np.arange(14.0, 30.01, 0.25)
scan = [(bp,) + dd_fit(wd, bp)[1:] for bp in grid]
sse_by_bp = pd.Series({bp: s for bp, s, _ in scan})
BP = float(sse_by_bp.idxmin())
COEF, _, R2 = dd_fit(wd, BP)
r_lin = float(np.corrcoef(wd.tmax, wd.peak)[0, 1])

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.10, column_widths=[0.66, 0.34],
                    subplot_titles=["Daily peak demand against daily maximum temperature",
                                    "Where the balance point is"])

# Points coloured by their own temperature, on the diverging scale centred at BP.
fig.add_trace(go.Scatter(
    x=wd.tmax, y=wd.peak, mode='markers', name='weekday',
    marker=dict(size=5.5, opacity=0.62, color=wd.tmax, colorscale=DIVERGING,
                cmin=BP - 16, cmax=BP + 16, line=dict(width=0)),
    hovertemplate="%{x:.1f} °C<br>%{y:,.0f} MW<extra></extra>", showlegend=False), row=1, col=1)

xs = np.linspace(wd.tmax.min(), wd.tmax.max(), 200)
ys = COEF[0] + COEF[1] * np.maximum(xs - BP, 0) + COEF[2] * np.maximum(BP - xs, 0)
fig.add_trace(go.Scatter(x=xs, y=ys, mode='lines', name='degree-day fit',
                         line=dict(color=INK, width=2.4),
                         hovertemplate="%{x:.1f} °C<br>fit %{y:,.0f} MW<extra></extra>"),
              row=1, col=1)

# The straight line that r = 0.27 describes, for contrast.
sl, ic = np.polyfit(wd.tmax, wd.peak, 1)
fig.add_trace(go.Scatter(x=xs, y=ic + sl * xs, mode='lines', name='linear fit',
                         line=dict(color=MUTED, width=1.8, dash='dash'),
                         hovertemplate="%{x:.1f} °C<br>linear %{y:,.0f} MW<extra></extra>"),
              row=1, col=1)

fig.add_vline(x=BP, line=dict(color=INK_2, width=1.2, dash='dot'), row=1, col=1)
fig.add_annotation(x=BP, y=wd.peak.max(), text=f" balance point {BP:.1f} °C", showarrow=False,
                   xanchor='left', font=dict(size=11.5, color=INK_2), row=1, col=1)
fig.add_annotation(
    x=wd.tmax.min(), y=wd.peak.max(), xanchor='left', showarrow=False,
    text=(f"<b>R² = {R2:.3f}</b> piecewise<br>"
          f"<span style='color:{MUTED}'>R² = {r_lin ** 2:.3f} linear (r = {r_lin:.2f})</span>"),
    align='left', font=dict(size=12.5, color=INK), row=1, col=1)

fig.add_trace(go.Scatter(x=sse_by_bp.index, y=sse_by_bp.values / 1e6, mode='lines',
                         line=dict(color=BLUE, width=2), showlegend=False,
                         hovertemplate="bp %{x:.2f} °C<br>SSE %{y:,.0f}M<extra></extra>"),
              row=1, col=2)
fig.add_vline(x=BP, line=dict(color=INK, width=1.4, dash='dot'),
              annotation_text=f" {BP:.1f} °C", annotation_position="top left",
              annotation_font=dict(size=11, color=INK), row=1, col=2)

fig.update_xaxes(title_text="max daily temperature (°C)", row=1, col=1)
fig.update_yaxes(title_text="daily peak operational demand (MW)", row=1, col=1)
fig.update_xaxes(title_text="candidate balance point (°C)", row=1, col=2)
fig.update_yaxes(title_text="sum of squared error (millions)", row=1, col=2)
style(fig, "Temperature explains peak demand — but not linearly",
      f"{len(wd)} weekdays. A straight line finds r = {r_lin:.2f} and concludes almost nothing. "
      f"Breaking it at {BP:.1f} °C lifts R² to {R2:.2f}.",
      height=500, hover='closest', legend_y=1.02)
fig.show()

print(f"balance point   {BP:.2f} °C   (searched {grid.min():.0f}-{grid.max():.0f} °C, "
      f"{len(grid)} candidates)")
print(f"base load       {COEF[0]:,.0f} MW at the balance point")
print(f"cooling slope  +{COEF[1]:.1f} MW per degree above")
print(f"heating slope  +{COEF[2]:.1f} MW per degree below")
print(f"R²              {R2:.3f}  (linear r² would be {r_lin ** 2:.3f})")
print(f"\ncold-arm support: {int((wd.tmax < 20).sum())} weekdays below 20 °C, "
      f"{int((wd.tmax < 15).sum())} below 15 °C — the heating slope is the softer of the two.")

balance point   27.00 °C   (searched 14-30 °C, 65 candidates)
base load       2,412 MW at the balance point
cooling slope  +91.9 MW per degree above
heating slope  +81.2 MW per degree below
R²              0.721  (linear r² would be 0.074)

cold-arm support: 140 weekdays below 20 °C, 4 below 15 °C — the heating slope is the softer of the two.


### 4. Heatwaves compound

A degree-day model has no memory: it gives the same answer for the first 38 degC day of a run and the
third. Buildings do not work that way. Thermal mass takes days to charge, overnight minima stay
high, and air conditioning that ran all night starts the next day behind.

The test controls for temperature rather than assuming it away. Restricting to weekdays in the
**35–40 degC band only**, so day one and day three are at essentially the same maximum, demand still
rises with position in the run. That difference is not the temperature, because the temperature is
held fixed; it is the accumulated heat.

The consequence for anyone using the chart-3 model: it will under-forecast late-heatwave peaks,
which are exactly the peaks that matter for reserve.

In [9]:
# ── Chart 4: position in a run of hot days, temperature held fixed ───────────
HOT = 35.0
hot = (day.tmax >= HOT).astype(int)
# Run length: consecutive calendar days at or above the threshold, counting from 1.
day['run'] = hot * (hot.groupby((hot != hot.shift()).cumsum()).cumcount() + 1)

band = day[(~day.is_weekend) & (day.tmax >= 35) & (day.tmax < 40)].copy()
band['pos'] = band.run.clip(upper=3)
bs = band.groupby('pos').agg(peak=('peak', 'mean'), tmax=('tmax', 'mean'),
                             n=('peak', 'size'))

allhot = day[(~day.is_weekend) & (day.tmax >= HOT)].copy()
allhot['pos'] = allhot.run.clip(upper=4)
as_ = allhot.groupby('pos').agg(peak=('peak', 'mean'), tmax=('tmax', 'mean'),
                                n=('peak', 'size'))

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.11,
                    subplot_titles=[f"Weekdays in the 35–40 °C band only",
                                    f"All weekdays ≥ {HOT:.0f} °C (temperature not held fixed)"])

for col, frame, cap in ((1, bs, 3), (2, as_, 4)):
    labels = [f"day {int(i)}" + ("+" if i == cap else "") for i in frame.index]
    fig.add_trace(go.Bar(
        x=labels, y=frame.peak, marker_color=ORANGE_RAMP[3], showlegend=False,
        text=[f"{v:,.0f} MW" for v in frame.peak], textposition='outside', cliponaxis=False,
        customdata=np.column_stack([frame.tmax, frame.n]),
        hovertemplate="%{x}<br>peak %{y:,.0f} MW<br>mean Tmax %{customdata[0]:.1f} °C"
                      "<br>n = %{customdata[1]}<extra></extra>"), row=1, col=col)
    # The mean temperature of each bar, printed under it: this is the control.
    for x, tv, nv in zip(labels, frame.tmax, frame.n):
        fig.add_annotation(x=x, y=0, yshift=-16, text=f"{tv:.1f} °C · n={int(nv)}",
                           showarrow=False, font=dict(size=10.5, color=MUTED), row=1, col=col)

lo, hi = bs.peak.iloc[0], bs.peak.iloc[-1]
fig.update_yaxes(title_text="mean daily peak demand (MW)", range=[0, as_.peak.max() * 1.18],
                 row=1, col=1)
fig.update_yaxes(range=[0, as_.peak.max() * 1.18], row=1, col=2)
style(fig, "The third hot day costs more than the first, at the same temperature",
      f"In the 35–40 °C band the mean maximum barely moves ({bs.tmax.iloc[0]:.1f} → "
      f"{bs.tmax.iloc[-1]:.1f} °C) while mean peak demand rises {hi - lo:,.0f} MW "
      f"({(hi / lo - 1) * 100:.0f}%).",
      height=470, hover='closest', bottom=76, showlegend=False)
fig.show()

print("35–40 °C band, weekdays — temperature held fixed, position in run varying:")
print(bs.round(1).to_string())
print(f"\nday 1 -> day 3+: {hi - lo:+,.0f} MW ({(hi / lo - 1) * 100:+.1f}%) "
      f"on a temperature change of {bs.tmax.iloc[-1] - bs.tmax.iloc[0]:+.1f} °C.")
print(f"the chart-3 model would predict a change of only "
      f"{COEF[1] * (bs.tmax.iloc[-1] - bs.tmax.iloc[0]):+,.0f} MW for that.")

35–40 °C band, weekdays — temperature held fixed, position in run varying:
       peak  tmax   n
pos                  
1    3212.6  36.9  32
2    3550.6  37.8  10
3    3559.2  37.2  28

day 1 -> day 3+: +347 MW (+10.8%) on a temperature change of +0.4 °C.
the chart-3 model would predict a change of only +34 MW for that.


### 5. Where the price tail lives

`docs/DATA_NOTES.md` quirk 9 records that the administered price cap changed: 738.00 in 2023–24,
then a higher ceiling from 2025. Any statement about upside prices that spans that change is partly
a statement about a rule. So this chart uses **2025–26 only**, where one cap applies throughout, and
the comparability checks re-run it on 2024 to show the direction survives.

Two panels. Spike *probability* on the left — the share of days whose maximum trading price exceeds
300 $/MWh — and the price distribution itself on the right. The story is entirely in the top bin.
Below 40 degC, temperature barely moves spike risk, and it is not even monotonic: mild days below
25 degC spike slightly more often than 30-35 degC days, because winter evening peaks are their own
event. Above 40 degC the rate jumps to roughly half of all days.

This is the practical payoff of the whole notebook: a variable that looked worthless at r = 0.27 is
in fact the single best available predictor of the days the market gets expensive.

In [10]:
# ── Chart 5: spike probability and price distribution by temperature ─────────
price = load_price(raw='../data/raw').set_index('ts')
pday = pd.DataFrame({'pmax': price.price.resample('D').max(),
                     'pmean': price.price.resample('D').mean()})
j = day.join(pday, how='inner').dropna(subset=['pmax'])

SPIKE = 300.0
BINS = [0, 25, 30, 35, 40, 60]
NAMES = ['<25', '25–30', '30–35', '35–40', '>40']
late = j[j.year >= 2025].copy()          # one cap regime throughout
late['bin'] = pd.cut(late.tmax, BINS, labels=NAMES)

sp = late.groupby('bin', observed=True).agg(
    rate=('pmax', lambda x: (x > SPIKE).mean()),
    mean_pmax=('pmax', 'mean'), n=('pmax', 'size'))

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.10, column_widths=[0.46, 0.54],
                    subplot_titles=[f"Share of days with a trading price above {SPIKE:.0f} $/MWh",
                                    "Daily maximum price by temperature band"])

hue = [temp_hue(u) for u in np.linspace(0.42, 1.0, len(sp))]
fig.add_trace(go.Bar(x=list(sp.index.astype(str)), y=sp.rate, marker_color=hue, showlegend=False,
                     text=[f"{v * 100:.0f}%" for v in sp.rate], textposition='outside',
                     cliponaxis=False, customdata=sp.n,
                     hovertemplate="%{x} °C<br>%{y:.1%} of days<br>n = %{customdata}<extra></extra>"),
              row=1, col=1)

for k, nm in enumerate(NAMES):
    v = late.loc[late.bin == nm, 'pmax'].dropna()
    if v.empty:
        continue
    fig.add_trace(go.Box(y=v, name=nm, marker_color=hue[k], line=dict(width=1.4),
                         boxpoints='outliers', marker=dict(size=4, opacity=0.6),
                         showlegend=False,
                         hovertemplate=f"{nm} °C<br>%{{y:,.0f}} $/MWh<extra></extra>"),
                  row=1, col=2)

fig.add_hline(y=SPIKE, line=dict(color=INK_2, width=1, dash='dot'), row=1, col=2)
fig.update_yaxes(title_text="share of days", tickformat='.0%', range=[0, 0.66], row=1, col=1)
fig.update_xaxes(title_text="max daily temperature (°C)", row=1, col=1)
fig.update_yaxes(title_text="daily max trading price ($/MWh)", row=1, col=2)
fig.update_xaxes(title_text="max daily temperature (°C)", row=1, col=2)
style(fig, "The price tail is a hot-day phenomenon, and only above 40 °C",
      f"2025–26 only, so one administered cap applies throughout ({len(late)} days). "
      f"Below 40 °C spike risk is flat at roughly {sp.rate.iloc[:-1].mean() * 100:.0f}%; "
      f"above it, {sp.rate.iloc[-1] * 100:.0f}%.",
      height=470, hover='closest', showlegend=False)
fig.show()

print("2025–26, by temperature band:")
print(sp.assign(rate=(sp.rate * 100).round(1)).round(1).to_string())
lift = sp.rate.iloc[-1] / sp.rate.iloc[:-1].mean()
print(f"\n>40 °C days are {lift:.1f}x as likely to spike as all cooler days combined.")

2025–26, by temperature band:
       rate  mean_pmax    n
bin                        
<25    16.6      216.9  301
25–30   9.2      163.5  120
30–35   8.8      155.1   91
35–40  14.5      202.0   69
>40    48.0      415.7   25

>40 °C days are 3.9x as likely to spike as all cooler days combined.


### 6. Storage on hot days — the hypothesis that did not survive

This dataset is the only one in the project that gives **per-facility** temperature at sites where
something else is also measured per facility: the seven grid-scale batteries from
`EDA_facility_scada`. That makes one specific question answerable that regional weather could not
answer — do batteries derate in the heat? Lithium cells lose usable power at high ambient
temperature, and if the fleet were thermally limited it would show as peak MW falling on hot days.

**It does not happen.** Peak discharge power on hot days is flat or *higher* than on mild days at
every mature unit — Collie ESR4 goes from 197 MW on days below 25 degC to 238 MW above 35 degC, and
no unit shows the falling profile derating would produce.

The honest reading is that this chart cannot see derating even if it exists, because dispatch is
market-driven: hot days are high-price days (chart 5), so the batteries are being asked for more
power at exactly the times thermal limits would bite. Market response swamps any thermal signal.
Separating them would need cell temperature or an availability declaration, and neither is in this
data. The finding is recorded as a **null result with a stated confound**, not as evidence of
absence.

In [11]:
# ── Chart 6: battery peak power against own-site temperature ─────────────────
sc = load_facility_scada(raw='../data/raw')
sc['date'] = sc.ts.dt.normalize()

tl = t[['date', 'facility_code', 'temp_max_c']]
bd = (sc.groupby(['facility_code', 'date'])
        .agg(pk=('mw', 'max'), dis=('Average MWh', lambda x: x[x > 0].sum()))
        .reset_index()
        .merge(tl, on=['date', 'facility_code'], how='inner'))

PMAX = sc.groupby('facility_code').mw.max()
# Mature only: before a unit reaches 90% of its eventual peak it is commissioning,
# and its power figures describe a test programme rather than a battery (SCADA nb).
mature_from = {f: sc.loc[(sc.facility_code == f) & (sc.mw >= 0.9 * PMAX[f]), 'ts'].min()
               for f in PMAX.index}
bd = bd[bd.apply(lambda r: r.date >= mature_from[r.facility_code], axis=1)]

BBINS, BNAMES = [0, 25, 30, 35, 60], ['<25', '25–30', '30–35', '>35']
bd['bin'] = pd.cut(bd.temp_max_c, BBINS, labels=BNAMES)
facs = [f for f in sorted(bd.facility_code.unique()) if (bd.facility_code == f).sum() >= 120]

fig = make_subplots(rows=2, cols=3, shared_yaxes=False, vertical_spacing=0.20,
                    horizontal_spacing=0.07,
                    subplot_titles=[f"{f}  ·  {PMAX[f]:.0f} MW" for f in facs[:6]])
hue = [temp_hue(u) for u in np.linspace(0.42, 1.0, len(BNAMES))]

for k, f in enumerate(facs[:6]):
    r, c = divmod(k, 3)
    g = bd[bd.facility_code == f].groupby('bin', observed=True).agg(
        pk=('pk', 'mean'), n=('pk', 'size'))
    fig.add_trace(go.Bar(x=list(g.index.astype(str)), y=g.pk, marker_color=hue[:len(g)],
                         showlegend=False, customdata=g.n,
                         hovertemplate="%{x} °C<br>peak %{y:.0f} MW<br>n = %{customdata}"
                                       "<extra></extra>"), row=r + 1, col=c + 1)
    fig.add_hline(y=PMAX[f], line=dict(color=MUTED, width=1, dash='dot'), row=r + 1, col=c + 1)
    fig.update_yaxes(range=[0, PMAX[f] * 1.12], row=r + 1, col=c + 1)

fig.update_yaxes(title_text="mean daily peak (MW)", row=1, col=1)
fig.update_yaxes(title_text="mean daily peak (MW)", row=2, col=1)
style(fig, "No sign of thermal derating — and the chart cannot rule it out either",
      "Mean daily peak discharge by own-site temperature, mature operation only. Dotted line is "
      "each unit's measured maximum. Hot days are high-price days, so market response and thermal "
      "limits are confounded.",
      height=620, hover='closest', showlegend=False)
fig.show()

chk = (bd.groupby(['facility_code', 'bin'], observed=True).pk.mean().unstack())
print("mean daily peak discharge (MW) by own-site temperature band:")
print(chk.round(1).to_string())

mean daily peak discharge (MW) by own-site temperature band:
bin                <25  25–30  30–35    >35
facility_code                              
ALINTA_WGP_ESR1    5.9    NaN    NaN    NaN
COLLIE_BESS2     227.1  217.1  232.2  252.6
COLLIE_ESR1      159.5  156.9  153.9  165.6
COLLIE_ESR4      196.9  184.4  200.8  237.8
COLLIE_ESR5      208.8  209.0  223.9  240.8
KWINANA_ESR1      84.2   80.8   82.1   90.4
KWINANA_ESR2     197.4  153.3  162.7  183.6


## Comparability checks

Four things could each have manufactured the results above.

1. **The reference station.** Every demand result uses one station as a proxy for "Perth weather".
   If the balance point is an artefact of that choice, refitting on the other 20 stations will move
   it around.
2. **Weekday selection.** The fit excludes weekends. If temperature sensitivity is really a
   day-of-week effect in disguise, the weekend fit will differ in slope, not just in level.
3. **The trend.** Demand grew over three years and 2026 is only eight months. A year-by-year refit
   separates the weather response from the trend.
4. **The cap change.** Chart 5 used 2025–26 only. The 2024 data has a lower cap, which censors the
   upside — so the spike rate there should be *lower*, and the temperature ordering should survive.

In [12]:
# ── Comparability checks ─────────────────────────────────────────────────────
print("1. REFERENCE STATION — this is the weakest link in the chart-3 result.")
rows = []
for s in wst.columns:
    f = pd.DataFrame({'tmax': wst[s], 'peak': day.peak}).dropna()
    f = f[f.index.dayofweek < 5]
    if len(f) < 300:
        continue
    sc_ = pd.Series({bp: dd_fit(f, bp)[1] for bp in grid})
    bp_s = float(sc_.idxmin())
    rows.append(dict(station=s, n=len(f), bp=bp_s, r2=dd_fit(f, bp_s)[2]))
rs = pd.DataFrame(rows).sort_values('r2', ascending=False)
print(f"   {len(rs)} stations refitted independently.")
print(f"   balance point: median {rs.bp.median():.2f} °C, "
      f"IQR {rs.bp.quantile(.25):.2f}–{rs.bp.quantile(.75):.2f}, "
      f"range {rs.bp.min():.2f}–{rs.bp.max():.2f}")
print(f"   R²: median {rs.r2.median():.3f}, best {rs.r2.max():.3f} ({rs.station.iloc[0]})")
print(f"   reference station {REF_STATION}: bp {BP:.2f} °C, R² {R2:.3f} "
      f"-> rank {int((rs.r2 > R2).sum()) + 1} of {len(rs)} by fit quality")
print( "   VERDICT: the U-shape is universal - every station fits far better than a line - but")
print(f"   the balance point is NOT stable at the quarter-degree precision the search reports.")
print(f"   It ranges {rs.bp.min():.1f}-{rs.bp.max():.1f} °C depending on which station stands in")
print(f"   for 'Perth weather', and the reference station sits at the top of that range. The")
print(f"   defensible statement is a balance point of roughly {rs.bp.median():.0f} °C, not "
      f"{BP:.2f} °C.")

print("\n2. WEEKDAYS vs WEEKENDS — same shape, lower level. Not a day-of-week effect.")
we = day[day.is_weekend]
c_we, _, r2_we = dd_fit(we, BP)
print(f"   weekday: base {COEF[0]:,.0f} MW, cooling +{COEF[1]:.1f}, heating +{COEF[2]:.1f} MW/°C, "
      f"R² {R2:.3f}")
print(f"   weekend: base {c_we[0]:,.0f} MW, cooling +{c_we[1]:.1f}, heating +{c_we[2]:.1f} MW/°C, "
      f"R² {r2_we:.3f}")
print(f"   the slopes agree within {abs(COEF[1] - c_we[1]) / COEF[1] * 100:.0f}% (cooling); "
      f"the base differs by {COEF[0] - c_we[0]:,.0f} MW, which is the day-of-week effect.")

print("\n3. YEAR BY YEAR — the response is stable; the base load is what moves.")
yr = []
for y in YEARS:
    f = wd[wd.year == y]
    c_, _, r2_ = dd_fit(f, BP)
    yr.append(dict(year=y, days=len(f), base=c_[0], cooling=c_[1], heating=c_[2], r2=r2_))
yrs = pd.DataFrame(yr).set_index('year')
print(yrs.round(2).to_string())
print("   note 2026 is January-August only, so it carries more summer than the full years.")

print("\n4. CAP CHANGE — a fixed $300 line is NOT comparable across the two cap regimes.")
# Under the 738.00 cap, 300 $/MWh was an ordinary price rather than a spike: judged on
# that fixed line, 2024 "spikes" on 55-75% of days in EVERY temperature band, which says
# nothing about temperature. The comparable question is whether hot days land in the top
# decile OF THEIR OWN ERA, so each regime is judged against its own distribution. The
# threshold is inclusive because 2024's decile IS the cap, and > would return zero.
j2 = j.copy()
j2['era'] = np.where(j2.year <= 2024, '2024', '2025-26')
j2['bin'] = pd.cut(j2.tmax, BINS, labels=NAMES)
out = {}
for era, g in j2.groupby('era'):
    thr = g.pmax.quantile(0.90)
    hi = g.assign(hi=g.pmax >= thr).groupby('bin', observed=True).hi.agg(['mean', 'size'])
    out[f'{era} top-decile %'] = (hi['mean'] * 100).round(1)
    out[f'{era} n'] = hi['size']
    print(f"   {era}: top decile begins at {thr:,.0f} $/MWh "
          f"({(g.pmax >= thr).mean() * 100:.1f}% of days)")
print(pd.DataFrame(out).to_string())
print("   The top temperature bin has the highest rate in BOTH regimes, so the temperature")
print("   ordering is not an artefact of the cap change. It does mean the fixed-$300 view in")
print("   chart 5 is valid only within 2025-26 - which is the window chart 5 restricts itself to.")

1. REFERENCE STATION — this is the weakest link in the chart-3 result.
   18 stations refitted independently.
   balance point: median 25.25 °C, IQR 24.56–26.19, range 23.25–27.25
   R²: median 0.659, best 0.731 (S09_NEWGEN_KWINANA_CCG1)
   reference station S01_NEWGEN_NEERABUP_GT1: bp 27.00 °C, R² 0.721 -> rank 2 of 18 by fit quality
   VERDICT: the U-shape is universal - every station fits far better than a line - but
   the balance point is NOT stable at the quarter-degree precision the search reports.
   It ranges 23.2-27.2 °C depending on which station stands in
   for 'Perth weather', and the reference station sits at the top of that range. The
   defensible statement is a balance point of roughly 25 °C, not 27.00 °C.

2. WEEKDAYS vs WEEKENDS — same shape, lower level. Not a day-of-week effect.
   weekday: base 2,412 MW, cooling +91.9, heating +81.2 MW/°C, R² 0.721
   weekend: base 2,258 MW, cooling +87.8, heating +81.1 MW/°C, R² 0.709
   the slopes agree within 5% (cooling); the

## What this notebook establishes

- **Nine of 61 facility codes report a registered constant, not a temperature.** Six sit at exactly
  41.0 °C on all 973 days; three report a −1.401/−1.64 sentinel on 968 of 973. They are numeric and
  non-null, so a null check passes them, and including them shifts the fleet mean by −1.4 °C. The
  test that identifies them uses no magic values: a real SWIS series correlates 0.797 to 0.985 with
  the fleet-wide daily median, the sentinel scores −0.096, and the constants have no variance at all.
  `load_temperature` drops them by default. The fleet *mean* barely moves when they are included
  (+0.26 °C) — but only because 41.0 pulls up while −1.4 pulls down. Any per-facility or per-station
  figure takes the full error: `YANDIN_WF1` reports a 41.0 °C mean and `MUNGARRA_GT1` a −1.36 °C one,
  against a fleet median of 24.5 °C.
- **Fifty real facility series are twenty-one weather stations.** Ten facility codes share one
  Pinjar/Neerabup signal, identical to ten decimal places. Any analysis that treats facility columns
  as independent observations multiply-counts a handful of instruments; `temperature_stations`
  collapses them.
- **The linear correlation between temperature and peak demand is 0.27, and it is the wrong
  measurement.** The relationship is a U, not a line. Fitted as degree days on the reference station
  it reaches **R² = 0.72** against a linear r² of 0.07, with +91.9 MW per degree of cooling and
  +81.2 MW per degree of heating either side of a balance point of 27.0 °C.
- **The U-shape is universal; the balance point is not.** Refitting independently on each of the 18
  stations with enough coverage, every one prefers the piecewise form (median R² 0.66) — but the
  balance point ranges **23.3 to 27.3 °C**, median 25.3 °C, and the reference station sits at the top
  of that range. The defensible claim is a balance point of **roughly 25 °C** — still far from the
  18 °C of northern-hemisphere convention, but not the quarter-degree figure a single fit appears to
  offer. The slopes are the robust part: they agree within 5% across weekdays, weekends and all three
  years.
- **Heatwaves compound, and a degree-day model cannot see it.** Holding temperature inside the
  35–40 °C band, mean weekday peak rises from 3,213 MW on the first hot day to 3,559 MW on the third
  — **+347 MW, about +11%**, on a temperature change of +0.4 °C, for which the degree-day model would
  predict +34 MW. Any forecast built on that model alone will under-predict exactly the late-heatwave
  peaks that set reserve requirements. The caveat is sample size: 28 day-3+ weekdays in three years.
- **The price tail is a hot-day phenomenon, and it is a cliff rather than a slope.** On 2025–26 data,
  under one administered cap, days above 40 °C exceed 300 $/MWh on **48%** of occasions against
  roughly 12% across every cooler band — a 3.9x lift concentrated entirely in the top bin. Below
  40 °C temperature barely moves spike risk and is not even monotonic. Judged instead against each
  era's own top decile, so the 2024 cap change cannot manufacture the result, the hottest bin ranks
  highest in both regimes (35% and 40%).
- **No thermal derating is visible in the battery fleet, and this data cannot prove its absence.**
  Peak discharge on hot days is flat or higher at every mature unit — Collie ESR4 runs 197 MW below
  25 °C and 238 MW above 35 °C. Hot days are high-price days, so
  market response and thermal limits are confounded; separating them needs cell temperature or
  availability declarations, which AEMO does not publish here.

**Where this stops.** Daily maxima cannot reach the 5-minute demand series, so peak *timing*,
temperature-driven ramp rates and the overnight minima that actually drive multi-day heatwave load
are all out of scope. Reaching them means an external hourly source — the Bureau of Meteorology —
and that is a different dataset with a different licence, not a further slice of this one.